# M12 — PPO: Full 6000-Episode Budget Training Run (Colab T4)

This notebook trains the **PPO (M12)** continuous Gaussian baseline for a full paper-scale budget (**6000 episodes**, converted internally to **1,200,000 total environment steps** with `N_SLOTS=200`), matching PKTD3-TD's Table III `M_EPISODES`.

**Architecture & Training:**
- Continuous action space matching PKTD3-TD: 3D physical velocity $(v, \lambda, \rho)$ via $[-1, 1]^3$ normalized actions
- Separate Actor (Gaussian policy with learnable log-std) and Critic (state-value $V(s)$ network)
- Generalized Advantage Estimation (GAE-Lambda=0.95)
- PPO-Clip surrogate objective (clip_eps=0.2) with entropy bonus (coef=0.01)
- 2,048-step rollouts, 10 update epochs with minibatch size 64

**Workflow pattern:**
1. Cloned fresh to local Colab SSD (`/content/uav_trajectory_rl`) for fast I/O.
2. Checkpoints saved directly to **Google Drive** for persistent safety against disconnects.
3. Checkpoints saved every 50,000 steps (24 total checkpoints, matching the 250-episode cadence of run4).
4. At the end, 30-seed deterministic evaluation is run, checkpoints copied to repo, and pushed to GitHub using Colab's `GITHUB_PAT_TOKEN` secret.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone the repo fresh (local disk) and install

In [2]:
# Clone repo fresh to local disk with authenticated URL for seamless push at the end
import os
from google.colab import userdata

try:
    github_token = userdata.get('GITHUB_PAT_TOKEN')
except Exception:
    github_token = None

repo_owner = "Krishna200608"
repo_name = "uav_trajectory_rl"

if github_token:
    repo_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"
    print("Authenticated git URL configured using GITHUB_PAT_TOKEN secret.")
else:
    repo_url = f"https://github.com/{repo_owner}/{repo_name}.git"
    print("WARNING: GITHUB_PAT_TOKEN secret not found. Git push in Cell 8 will require manual auth.")

%cd /content
!rm -rf uav_trajectory_rl
!git clone {repo_url} uav_trajectory_rl
%cd uav_trajectory_rl
!pip install -e . --quiet
!git log --oneline -n 5

Authenticated git URL configured using GITHUB_PAT_TOKEN secret.
/content
Cloning into 'uav_trajectory_rl'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 553 (delta 36), reused 67 (delta 28), pack-reused 471 (from 1)
Receiving objects: 100% (553/553), 177.80 MiB | 37.26 MiB/s, done.
Resolving deltas: 100% (303/303), done.
Updating files: 100% (133/133), done.
/content/uav_trajectory_rl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for uav-trajectory-rl (pyproject.toml) ... done
de043b2 (HEAD -> main, origin/main, origin/HEAD) Update README.md with Dueling DQL and PPO Colab training notebooks
a2fff2a Add and calibrate Colab training notebooks for Dueling DQL and PPO baselines
b309457 Document Greedy's term

## 3. Confirm GPU and check train_ppo.py's actual CLI flags

Do not assume the flags below are exactly right — confirm against this output first.

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print()
!python scripts/train_ppo.py --help

CUDA available: False
Device: CPU only

usage: train_ppo.py [-h] [--total-steps TOTAL_STEPS] [--episodes EPISODES]
                    [--k K] [--seed SEED] [--rollout-length ROLLOUT_LENGTH]
                    [--checkpoint-dir CHECKPOINT_DIR]
                    [--checkpoint-every CHECKPOINT_EVERY]
                    [--log-every LOG_EVERY] [--lr LR] [--gamma GAMMA]
                    [--gae-lambda GAE_LAMBDA] [--clip-eps CLIP_EPS]
                    [--value-coef VALUE_COEF] [--entropy-coef ENTROPY_COEF]
                    [--update-epochs UPDATE_EPOCHS]
                    [--minibatch-size MINIBATCH_SIZE] [--no-progress-bar]

Train PPO Baseline for 3D UAV Trajectory Design

options:
  -h, --help            show this help message and exit
  --total-steps TOTAL_STEPS
                        Total environment steps (primary budget)
  --episodes EPISODES   Episode budget (converted to steps via N_SLOTS)
  --k K                 Number of ground users (default: 10)
  --seed SEED   

## 4. Set the Drive checkpoint path

In [4]:
import os
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/ppo_run1"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be written to:', DRIVE_CHECKPOINT_DIR)

Checkpoints will be written to: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/ppo_run1


## 5. Launch the full training run

Budget: `--episodes 6000` automatically sets `total_steps = 1,200,000` (6000 episodes × 200 slots), exactly matching PKTD3-TD's training exposure. With `rollout_length=2048`, this is ~586 rollout collection & update cycles.

Saving checkpoints every `50000` steps (`--checkpoint-every 50000`) produces **24 checkpoints across the run** (matching the 250-episode cadence of `run4` and Dueling DQL).

**This cell will run for a while — let it complete.**

In [5]:
!python scripts/train_ppo.py \
  --episodes 6000 \
  --seed 0 \
  --rollout-length 2048 \
  --checkpoint-dir "{DRIVE_CHECKPOINT_DIR}" \
  --checkpoint-every 50000 \
  --log-every 50

STARTING PPO BASELINE TRAINING (M12)
Total steps: 1200000 | K: 10 | Seed: 0
Rollout length: 2048 | LR: 0.0001 | gamma: 0.96
GAE-lambda: 0.95 | clip_eps: 0.2 | update_epochs: 10
Checkpoint Dir: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/ppo_run1
Episode   50 | step=  10000 | reward=+745.024 | avg(50)=+756.946 | steps_this_ep=200
Episode  100 | step=  20000 | reward=+829.266 | avg(50)=+862.608 | steps_this_ep=200
Episode  150 | step=  30000 | reward=+708.597 | avg(50)=+911.648 | steps_this_ep=200
Episode  200 | step=  40000 | reward=+934.303 | avg(50)=+869.068 | steps_this_ep=200
Episode  250 | step=  50000 | reward=+1004.545 | avg(50)=+1026.113 | steps_this_ep=200
Episode  300 | step=  60000 | reward=+1097.093 | avg(50)=+1042.363 | steps_this_ep=200
Episode  350 | step=  70000 | reward=+1357.940 | avg(50)=+1127.718 | steps_this_ep=200
Episode  400 | step=  80000 | reward=+1181.485 | avg(50)=+1088.404 | steps_this_ep=200
Episode  450 | step=  90000 | reward=+859.332 | 

## 6. Sanity-check the run completed properly

In [6]:
import glob, os, numpy as np

ckpts = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
print(f'Found {len(ckpts)} checkpoint files in Drive:')
for c in ckpts[-5:]:
    print(' ', os.path.basename(c))

rewards_path = f"{DRIVE_CHECKPOINT_DIR}/episode_rewards.npy"
if os.path.exists(rewards_path):
    r = np.load(rewards_path)
    print(f'\nTotal episodes logged: {len(r)}')
    print(f'First 100 mean reward: {r[:100].mean():.2f}')
    print(f'Last 100 mean reward:  {r[-100:].mean():.2f}')
    print(f'Max episode reward:    {r.max():.2f}')
else:
    print('WARNING: episode_rewards.npy not found -- check the run completed correctly.')

Found 24 checkpoint files in Drive:
  ppo_step768000.pt
  ppo_step819200.pt
  ppo_step870400.pt
  ppo_step921600.pt
  ppo_step972800.pt

Total episodes logged: 6001
First 100 mean reward: 809.78
Last 100 mean reward:  1460.19
Max episode reward:    1874.81


## 7. Behavioral check — arrival rate and entropy trend, not just reward

The 800-episode diagnostic showed PPO covering ~700m of the ~848m diagonal but 0% arrival — check whether the full budget closes that final gap, and whether policy entropy has genuinely decreased (committed policy) rather than staying high (still mostly random).

In [9]:
import glob, os, sys, torch
import numpy as np

# 1. Clean out the poisoned/cached namespace from sys.modules
for mod in list(sys.modules.keys()):
    if mod == 'uav_trajectory_rl' or mod.startswith('uav_trajectory_rl.'):
        del sys.modules[mod]

# 2. Dynamically locate the actual src/ directory containing uav_trajectory_rl
src_candidates = [
    '/content/uav_trajectory_rl/src',
    os.path.abspath('src'),
    os.path.abspath('.'),
    '/content/src',
    '/content/drive/MyDrive/uav_trajectory_rl/src',
    '/content/drive/MyDrive/Uav_trajectory_rl/src',
]

src_path = None
for p_dir in src_candidates:
    if os.path.exists(os.path.join(p_dir, 'uav_trajectory_rl', 'mdp_environment.py')):
        src_path = p_dir
        break

if not src_path:
    matches = glob.glob('/content/**/uav_trajectory_rl/mdp_environment.py', recursive=True)
    if matches:
        src_path = os.path.dirname(os.path.dirname(matches[0]))

if src_path:
    print('Package source located at:', src_path)
    while src_path in sys.path:
        sys.path.remove(src_path)
    sys.path.insert(0, src_path)
else:
    raise FileNotFoundError('Could not locate uav_trajectory_rl package source on disk. Please confirm Step 2 cloned successfully.')

from uav_trajectory_rl.mdp_environment import UAVTrajectoryEnv
from uav_trajectory_rl.baselines.ppo import PPOAgent

# Locate final checkpoint
final_ckpt_candidates = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*final*.pt")) or sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*.pt"))
if not final_ckpt_candidates:
    raise FileNotFoundError(f"No checkpoint files found in {DRIVE_CHECKPOINT_DIR}")
final_ckpt = final_ckpt_candidates[-1]
print('Evaluating checkpoint:', final_ckpt)

env_tmp = UAVTrajectoryEnv(k=10, rng=np.random.default_rng(0))
agent = PPOAgent(state_dim=env_tmp.state_dim)
agent.load(final_ckpt)
agent.actor.eval()
agent.critic.eval()

def rollout(seed, k=10):
    env = UAVTrajectoryEnv(k=k, rng=np.random.default_rng(seed))
    state = env.reset()
    start = env.uav_pos.copy()
    max_dist, done, steps, ep_reward, arrived = 0.0, False, 0, 0.0, False
    while not done and steps < 200:
        # NOTE: select_action_deterministic() internally evaluates the actor Gaussian mean,
        # clips to [-c, c], and unnormalizes to physical (v, lam, rho) directly.
        physical_action = agent.select_action_deterministic(state)
        state, r, done, info = env.step(physical_action)
        max_dist = max(max_dist, float(np.linalg.norm(env.uav_pos - start)))
        ep_reward += r
        steps += 1
        arrived = info.get('arrived', False)
    return max_dist, ep_reward, arrived, steps

dists, rewards, arrivals, steps_taken = [], [], [], []
for seed in range(30):
    d, r, a, s = rollout(seed)
    dists.append(d); rewards.append(r); arrivals.append(a); steps_taken.append(s)
dists = np.array(dists)

print(f'\n30-seed evaluation (deterministic mean action):')
print(f'  Mean max displacement:   {dists.mean():.1f} m')
print(f'  Median max displacement: {np.median(dists):.1f} m')
print(f'  Frac > 50m:              {(dists > 50).mean():.1%}')
print(f'  ARRIVAL RATE:            {np.mean(arrivals):.1%} ({sum(arrivals)}/30)')
print(f'  Mean episode reward:     {np.mean(rewards):.2f}')
print(f'  Mean steps taken:        {np.mean(steps_taken):.1f}')


Package source located at: /content/uav_trajectory_rl/src
Evaluating checkpoint: /content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/ppo_run1/ppo_final.pt

30-seed evaluation (deterministic mean action):
  Mean max displacement:   701.9 m
  Median max displacement: 721.8 m
  Frac > 50m:              100.0%
  ARRIVAL RATE:            0.0% (0/30)
  Mean episode reward:     1469.63
  Mean steps taken:        200.0


## 8. Copy results into the local repo clone and push

**Run this only after confirming Cells 6-7 look reasonable.**

In [10]:
import shutil

LOCAL_CHECKPOINT_DIR = "/content/uav_trajectory_rl/checkpoints/ppo_run1"
os.makedirs(LOCAL_CHECKPOINT_DIR, exist_ok=True)

for f in glob.glob(f"{DRIVE_CHECKPOINT_DIR}/*"):
    shutil.copy(f, LOCAL_CHECKPOINT_DIR)

print('Copied files:')
!ls -la "{LOCAL_CHECKPOINT_DIR}"

Copied files:
total 42496
drwxr-xr-x 2 root root    4096 Aug 31 07:55 .
drwxr-xr-x 7 root root    4096 Aug 31 07:55 ..
-rw------- 1 root root   24132 Aug 31 07:55 episode_rewards.npy
-rw------- 1 root root  777697 Aug 31 07:55 episode_stats.json
-rw------- 1 root root 1772731 Aug 31 07:55 ppo_final.pt
-rw------- 1 root root  138671 Aug 31 07:55 ppo_reward_curve.png
-rw------- 1 root root 1773143 Aug 31 07:55 ppo_step1024000.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step102400.pt
-rw------- 1 root root 1773143 Aug 31 07:55 ppo_step1075200.pt
-rw------- 1 root root 1773143 Aug 31 07:55 ppo_step1126400.pt
-rw------- 1 root root 1773143 Aug 31 07:55 ppo_step1177600.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step153600.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step204800.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step256000.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step307200.pt
-rw------- 1 root root 1773085 Aug 31 07:55 ppo_step358400.pt
-rw-----

In [11]:
%cd /content/uav_trajectory_rl
!git config user.email "krishnasikheriya001@gmail.com"
!git config user.name "Krishna200608"

# Ensure remote URL has the token for non-interactive push
import os
if github_token:
    os.system(f"git remote set-url origin {repo_url}")

!git add -f checkpoints/ppo_run1
!git status
!git commit -m "Add PPO (M12) full training run and checkpoints"
!git push origin main
!git log --oneline -n 3

/content/uav_trajectory_rl
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   checkpoints/ppo_run1/episode_rewards.npy
	new file:   checkpoints/ppo_run1/episode_stats.json
	new file:   checkpoints/ppo_run1/ppo_final.pt
	new file:   checkpoints/ppo_run1/ppo_reward_curve.png
	new file:   checkpoints/ppo_run1/ppo_step102400.pt
	new file:   checkpoints/ppo_run1/ppo_step1024000.pt
	new file:   checkpoints/ppo_run1/ppo_step1075200.pt
	new file:   checkpoints/ppo_run1/ppo_step1126400.pt
	new file:   checkpoints/ppo_run1/ppo_step1177600.pt
	new file:   checkpoints/ppo_run1/ppo_step153600.pt
	new file:   checkpoints/ppo_run1/ppo_step204800.pt
	new file:   checkpoints/ppo_run1/ppo_step256000.pt
	new file:   checkpoints/ppo_run1/ppo_step307200.pt
	new file:   checkpoints/ppo_run1/ppo_step358400.pt
	new file:   checkpoints/ppo_run1/ppo_step409600.pt
	new file:   checkpoints/ppo_run1/ppo_step460800.

In [13]:
%cd /content/uav_trajectory_rl
!git pull --rebase origin main
!git push origin main
!git log --oneline -n 5


/content/uav_trajectory_rl
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 15 (delta 14), reused 9 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 3.84 KiB | 655.00 KiB/s, done.
From https://github.com/Krishna200608/uav_trajectory_rl
 * branch            main       -> FETCH_HEAD
   de043b2..d268034  main       -> origin/main
Successfully rebased and updated refs/heads/main.
Enumerating objects: 33, done.
Counting objects: 100% (33/33), done.
Delta compression using up to 2 threads
Compressing objects: 100% (31/31), done.
Writing objects: 100% (31/31), 35.57 MiB | 13.10 MiB/s, done.
Total 31 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Krishna200608/uav_trajectory_rl.git
   d268034..24d87d3  main -> main
24d87d3 (HEAD -> main, origin/main, origin/HEAD) Add PPO (M12) full training r

## Done

Bring the results (Step 7's evaluation numbers, and the pushed commit hash) back to the main conversation for independent review before this is used in M14's comparison plots.